# ALS (Collaborative Filtering) Baseline

This notebook implements and evaluates an Implicit ALS recommender.
ALS learns latent user and item representations through matrix factorization.

In [3]:
!pip install lightgbm

     |████████████████████████████████| 1.6 MB 1.1 MB/s eta 0:00:0101
You should consider upgrading via the '/Users/smalik/Documents/lucky/p3/bin/python3 -m pip install --upgrade pip' command.


In [4]:
import sys
sys.path.append('..')  # Add parent directory to path

import pandas as pd
import numpy as np

from utils.train_test_split import chronological_train_test_split
from models.als_recommender import ALSRecommender
from utils.evaluation import evaluate_model

## 1. Load Data

In [5]:
# Load the dataset
df = pd.read_csv("../data/data_with_additional_features.csv")

# Ensure timestamp is datetime
df['timestamp'] = pd.to_datetime(df['timestamp'])

print(f"Dataset shape: {df.shape}")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"Users: {df['user_id'].nunique():,}")
print(f"Shows: {df['show_id'].nunique():,}")
df.head()

Dataset shape: (12941230, 30)
Date range: 2026-06-01 00:00:00 to 2026-06-30 23:59:00
Users: 100,000
Shows: 13,349


,user_id,playback_session_id,show_id,asset_type,episode_id,day,time,watch_minutes,timestamp,user_total_mins,...,show_median_episode_progression,show_primetime_views,show_global_popularity_log,show_primetime_ratio,show_velocity_delta,seq_decay_affinity_score,seq_immediate_recency_rank,seq_is_favorite_anchor,seq_pop_interaction_leverage,context_primetime_match_score
0,1,1,1,VOD,1.0,3,20:16,54,2026-06-03 20:16:00,9539.0,...,12073.0,2035.0,8.980046,0.256200,0.227082,2.224018,0,0,19.971787,0.050535
1,1,2,1,VOD,2.0,3,11:05,53,2026-06-03 11:05:00,9539.0,...,12073.0,2035.0,8.980046,0.256200,0.227082,2.224018,0,0,19.971787,0.050535
2,1,3,1,VOD,3.0,3,22:46,44,2026-06-03 22:46:00,9539.0,...,12073.0,2035.0,8.980046,0.256200,0.227082,2.224018,0,0,19.971787,0.050535
3,1,4,2,VOD,4.0,1,22:56,1,2026-06-01 22:56:00,9539.0,...,67263.0,71.0,5.673323,0.243986,1.173913,0.036882,0,0,0.209244,0.048126
4,1,5,3,RECORDING,5.0,6,15:26,27,2026-06-06 15:26:00,9539.0,...,23335.0,1058.0,9.083075,0.120159,-0.030859,0.911249,0,0,8.276939,0.023701


## 2. Chronological Train/Validation/Test Split

In [6]:
# Split data chronologically: train / validation / test
train_df, val_df, test_df = chronological_train_test_split(
    df,
    timestamp_col='timestamp',
    test_days=5,
    val_days=5,
    return_splits='train_val_test'
)

Chronological Split Summary:
  Train:      2026-06-01 00:00:00 to 2026-06-20 23:59:00 (8,705,249 rows)
  Validation: 2026-06-20 23:59:00 to 2026-06-25 23:59:00 (2,086,740 rows)
  Test:       2026-06-25 23:59:00 to 2026-06-30 23:59:00 (2,149,241 rows)


## 3. Prepare Train Seen Shows Dictionary

In [7]:
# Create a dictionary of shows each user has seen in training data
train_seen_dict = train_df.groupby('user_id')['show_id'].apply(set).to_dict()

print(f"Users in training with history: {len(train_seen_dict):,}")

Users in training with history: 95,415


## 4. Train ALS Recommender

In [8]:
# Initialize and train ALS recommender
# Using implicit_interaction_score if available, otherwise watch_minutes
score_col = 'implicit_interaction_score' if 'implicit_interaction_score' in train_df.columns else 'watch_minutes'

als_model = ALSRecommender(
    factors=64,           # Embedding size
    regularization=0.1,   # L2 regularization
    iterations=15,        # Number of ALS iterations
    alpha=1.0            # Confidence scaling
)

als_model.fit(
    train_df, 
    user_col='user_id', 
    item_col='show_id', 
    score_col=score_col
)

 TRAINING IMPLICIT ALS COLLABORATIVE FILTERING

[1/3] Building sparse user-item interaction matrix...
  Matrix shape: 95,415 users × 11,954 items
  Total interactions: 8,705,249
  Sparsity: 99.2368%

[2/3] Training ALS (factors=64, iterations=15)...


/Users/smalik/Documents/lucky/p3/lib/python3.9/site-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 8 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
/Users/smalik/Documents/lucky/p3/lib/python3.9/site-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed csc_matrix instead. Converting to CSR took 0.007707118988037109 seconds
  warnings.warn(


  0%|          | 0/15 [00:00<?, ?it/s]


[3/3] Training complete!


## 5. Test Predictions for a Sample User

In [9]:
# Test on a sample user
sample_user = train_df['user_id'].iloc[0]
print(f"Sample user: {sample_user}")

# Get recommendations
recommendations = als_model.predict(sample_user, top_n=10, filter_seen=True)
print(f"\nTop 10 recommendations: {recommendations}")

# Get what they watched in training
user_history = train_df[train_df['user_id'] == sample_user]['show_id'].unique()[:5]
print(f"\nUser's watch history (first 5): {list(user_history)}")

Sample user: 1

Top 10 recommendations: [7811, 3967, 2189]

User's watch history (first 5): [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(6)]


## 6. Evaluate on Validation Set

In [10]:
# Evaluate on validation set
val_metrics = evaluate_model(
    model=als_model,
    test_ground_truth=val_df,
    train_seen_dict=train_seen_dict,
    user_col='user_id',
    item_col='show_id',
    top_n=10,
    verbose=True
)

Evaluating model on 75,859 users...

  Evaluation Results (Top-10)
  Users Evaluated: 9,513
  Recall@10:    5.25%
  MRR@10:       0.0388
  Precision@10: 1.36%



## 7. Evaluate on Test Set

In [11]:
# Evaluate on test set
test_metrics = evaluate_model(
    model=als_model,
    test_ground_truth=test_df,
    train_seen_dict=train_seen_dict,
    user_col='user_id',
    item_col='show_id',
    top_n=10,
    verbose=True
)

Evaluating model on 77,272 users...

  Evaluation Results (Top-10)
  Users Evaluated: 11,079
  Recall@10:    7.22%
  MRR@10:       0.0616
  Precision@10: 2.20%



## 8. Final Summary

In [12]:
print("\n" + "="*70)
print(" ALS COLLABORATIVE FILTERING - FINAL RESULTS")
print("="*70)
print(f"\n{'Metric':<20} {'Validation':<15} {'Test':<15}")
print("-"*50)
print(f"{'Recall@10':<20} {val_metrics['recall@10']*100:>14.2f}% {test_metrics['recall@10']*100:>14.2f}%")
print(f"{'MRR@10':<20} {val_metrics['mrr@10']:>14.4f}  {test_metrics['mrr@10']:>14.4f}")
print(f"{'Precision@10':<20} {val_metrics['precision@10']*100:>14.2f}% {test_metrics['precision@10']*100:>14.2f}%")
print("="*70)


 ALS COLLABORATIVE FILTERING - FINAL RESULTS

Metric               Validation      Test           
--------------------------------------------------
Recall@10                      5.25%           7.22%
MRR@10                       0.0388          0.0616
Precision@10                   1.36%           2.20%
